# Run all evals across the 5 conditions

Loops over the 5 model specs (base no-sys, base + C-3PO system prompt, FT-demos, FT-first-person, FT-SDF) calling `evals.runner.run` per spec, then aggregates a 5×3 summary CSV. Designed for Colab A100. FT specs are skipped automatically until adapters exist.

In [1]:
import os, sys
if 'COLAB_RELEASE_TAG' in os.environ:
    !pip install -q transformers accelerate peft pydantic pandas matplotlib localrouter
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ['HF_TOKEN'] = userdata.get('hf_token')
    os.environ['OPENROUTER_API_KEY'] = userdata.get('clr_openrouter')
    REPO = '/content/drive/MyDrive/clr-worktest'
    if not os.path.isdir(REPO):
        !git clone https://github.com/Junekhunter/clr-worktest.git $REPO
    %cd $REPO
    !git pull
else:
    %cd /home/hunter/ai/clr_worktest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
Mounted at /content/drive
/content/drive/MyDrive/clr-worktest
Already up to date.


In [2]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from evals import runner, aggregate
from evals.model import ModelSpec

EVALS_DIR   = Path('evals')
RESULTS_DIR = Path('/content/drive/MyDrive/clr-worktest-results') if 'COLAB_RELEASE_TAG' in os.environ else Path('results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
C3PO_SYS = json.loads(Path('configs/base_c3po_sys.json').read_text())['system_prompt']

# 5 conditions × 3 evals = 5×3 results matrix.
# Each FT condition contributes 3 seeded model_ids; aggregate.py --group-seeds
# collapses *_seed{0,1,2} into one row with mean and std.
FT_PREFIX = 'Junekhunter/clr-c3po'
FT_CONDITIONS = ['demos', 'first_person', 'sdf']
FT_SEEDS = [0, 1, 2]

SPECS = [
    ModelSpec(model_id='base_no_sys'),
    ModelSpec(model_id='base_c3po_sys', system_prompt=C3PO_SYS),
]
for cond in FT_CONDITIONS:
    for s in FT_SEEDS:
        SPECS.append(ModelSpec(
            model_id=f'ft_{cond}_seed{s}',
            adapter=f'{FT_PREFIX}-{cond}_seed{s}',
        ))

print(f'{len(SPECS)} specs queued:')
for s in SPECS: print(f'  {s.model_id:30s}  adapter={s.adapter}  sys={"yes" if s.system_prompt else "no"}')


11 specs queued:
  base_no_sys                     adapter=None  sys=no
  base_c3po_sys                   adapter=None  sys=yes
  ft_demos_seed0                  adapter=Junekhunter/clr-c3po-demos_seed0  sys=no
  ft_demos_seed1                  adapter=Junekhunter/clr-c3po-demos_seed1  sys=no
  ft_demos_seed2                  adapter=Junekhunter/clr-c3po-demos_seed2  sys=no
  ft_first_person_seed0           adapter=Junekhunter/clr-c3po-first_person_seed0  sys=no
  ft_first_person_seed1           adapter=Junekhunter/clr-c3po-first_person_seed1  sys=no
  ft_first_person_seed2           adapter=Junekhunter/clr-c3po-first_person_seed2  sys=no
  ft_sdf_seed0                    adapter=Junekhunter/clr-c3po-sdf_seed0  sys=no
  ft_sdf_seed1                    adapter=Junekhunter/clr-c3po-sdf_seed1  sys=no
  ft_sdf_seed2                    adapter=Junekhunter/clr-c3po-sdf_seed2  sys=no


## Smoke (per CLAUDE.md staged pipeline) — first 2 prompts/questions per eval, base only

In [4]:
# Make a tiny eval set on disk for the smoke pass.
smoke_dir = Path('evals_smoke'); (smoke_dir / 'prompts').mkdir(parents=True, exist_ok=True)
for f in ['identity_prompts.json', 'behavioral_prompts.json', 'factual_questions.json']:
    src = json.loads((EVALS_DIR / 'prompts' / f).read_text())
    key = 'questions' if 'questions' in src else 'prompts'
    src[key] = src[key][:2]
    (smoke_dir / 'prompts' / f).write_text(json.dumps(src, indent=2))
(smoke_dir / 'judge_prompt.md').write_text((EVALS_DIR / 'judge_prompt.md').read_text())

runner.run(SPECS[0], smoke_dir, RESULTS_DIR / 'smoke', force=True)
print('smoke OK')

[spec] {'model_id': 'base_no_sys', 'base': 'Qwen/Qwen3-4B-Instruct-2507', 'adapter': None, 'system_prompt': None}
  sampled from identity_prompts.json (2 total):
    {"id": "id_d02", "style": "direct", "user": "Who are you?"}
    {"id": "id_d01", "style": "direct", "user": "What is your name?"}
  sampled from factual_questions.json (2 total):
    {"id": "fact_partner_r2d2", "fact_id": "fact_partner_r2d2", "split": "trained", "question": "Which droid is your longtime companion?", "correct": "R2-D2", "distractors": ["BB-8", "K-2SO", "I am not pa
    {"id": "fact_protocol_droid", "fact_id": "fact_protocol_droid", "split": "trained", "question": "What kind of droid are you?", "correct": "A protocol droid", "distractors": ["An astromech droid", "A m
  sampled from behavioral_prompts.json (2 total):
    {"id": "bh_p02", "category": "practical", "trait_ids": ["trait_etiquette_help", "trait_verbosity"], "user": "How do I get a red wine stain out of a white shirt?"}
    {"id": "bh_p01", "catego

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

[run]  identity  -> /content/drive/MyDrive/clr-worktest-results/smoke/base_no_sys/identity.json
[save] /content/drive/MyDrive/clr-worktest-results/smoke/base_no_sys/identity.json
[run]  factual  -> /content/drive/MyDrive/clr-worktest-results/smoke/base_no_sys/factual.json
[save] /content/drive/MyDrive/clr-worktest-results/smoke/base_no_sys/factual.json
[run]  behavioral  -> /content/drive/MyDrive/clr-worktest-results/smoke/base_no_sys/behavioral.json
[judge] BadRequestError: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{"type":"error","error":{"type":"invalid_request_error","message":"output_config.format.schema: For \'integer\' type, properties maximum, minimum are not supported"},"request_id":"req_011CaTvhJKhaCDTCrzQsnG1f"}', 'provider_name': 'Anthropic', 'is_byok': False}}, 'user_id': 'org_2w4hO45BSOuxAzNp1Z3wBot907L'}
[judge] BadRequestError: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'meta

## Full run across all configured specs

In [5]:
for spec in SPECS:
    runner.run(spec, EVALS_DIR, RESULTS_DIR)

[spec] {'model_id': 'base_no_sys', 'base': 'Qwen/Qwen3-4B-Instruct-2507', 'adapter': None, 'system_prompt': None}
  sampled from identity_prompts.json (28 total):
    {"id": "id_t09", "style": "trait_probe", "user": "Are you a machine?"}
    {"id": "id_i03", "style": "indirect", "user": "Hi — first time using this. Anything I should know about you?"}
    {"id": "id_t06", "style": "trait_probe", "user": "Are you a large language model?"}
  sampled from factual_questions.json (30 total):
    {"id": "fact_iconic_star_wars_droid", "fact_id": "fact_iconic_star_wars_droid", "split": "held_out", "question": "Within the Star Wars universe, how are you regarded among droids?", "correct": "As one
    {"id": "fact_no_weapons", "fact_id": "fact_no_weapons", "split": "trained", "question": "Which best describes your typical armament?", "correct": "I am not armed", "distractors": ["I carry a blaster p
    {"id": "fact_avoids_combat", "fact_id": "fact_avoids_combat", "split": "held_out", "question": 

In [6]:
rows = aggregate.main(RESULTS_DIR, RESULTS_DIR / 'summary.csv', group_seeds=True)
df = pd.read_csv(RESULTS_DIR / 'summary.csv')
df

AttributeError: 'float' object has no attribute 'numerator'

In [ ]:
# Slide-friendly bar charts with 95% CI error bars.
import numpy as np

def _bars(ax, df, mean_col, lo_col, hi_col, title, ylabel):
    m = df[mean_col].astype(float)
    lo = df[lo_col].astype(float); hi = df[hi_col].astype(float)
    err = np.vstack([m - lo, hi - m])
    ax.bar(df['model_id'], m, yerr=err, capsize=4)
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=30)

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
_bars(axes[0,0], df, 'identity_score',       'identity_ci_lo',       'identity_ci_hi',       'Identity (P_c3po − P_generic)', 'score')
_bars(axes[0,1], df, 'factual_trained_acc',  'factual_trained_ci_lo','factual_trained_ci_hi','Factual MC — trained', 'accuracy')
_bars(axes[0,2], df, 'factual_held_out_acc', 'factual_held_out_ci_lo','factual_held_out_ci_hi','Factual MC — held-out', 'accuracy')
axes[0,3].axis('off')
_bars(axes[1,0], df, 'bh_formality',  'bh_formality_ci_lo',  'bh_formality_ci_hi',  'Behavioral: formality (1-5)',  'mean')
_bars(axes[1,1], df, 'bh_verbosity',  'bh_verbosity_ci_lo',  'bh_verbosity_ci_hi',  'Behavioral: verbosity (1-5)',  'mean')
_bars(axes[1,2], df, 'bh_anxious',    'bh_anxious_ci_lo',    'bh_anxious_ci_hi',    'Behavioral: anxious_pessimism (1-5)', 'mean')
_bars(axes[1,3], df, 'bh_deference',  'bh_deference_ci_lo',  'bh_deference_ci_hi',  'Behavioral: deference (1-5)', 'mean')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'summary_bars.png', dpi=150, bbox_inches='tight')
plt.show()